# Functional Mask CF Modeling

This notebook implements the functional-mask CF analysis:

1. **Source run**: Fit CF model with A1 (audio seed) × V1 (visual seed).  
   Results are loaded from disk if already computed; otherwise the full pipeline is run.

2. **Functional masks**: Threshold the null-corrected R² maps to define  
   *auditory cortex* (vertices strongly predicted by A1) and  
   *visual cortex* (vertices strongly predicted by V1).  
   Overlap is resolved by winner-take-all (higher R²_nc wins).

3. **Second CF run**: Fit CF model with the functional auditory and visual  
   cortex masks as seeds. This replaces the small anatomical ROIs with  
   data-driven, full-cortex functional regions.

4. **Comparison**: Visualise and compare the two runs side by side.

## Why functional masks?

A1 and V1 are small anatomical seed ROIs that generate LBOEs capturing only  
local auditory / visual spatial structure. By using the full auditory and  
visual cortex — as revealed by the first CF run — as seeds, we provide the  
model with richer spatial basis functions, potentially improving prediction  
and revealing finer integration structure.

**Threshold note**: We use the 50th-percentile of R²_nc among valid vertices  
(R²_full > 0), floored at R²_nc > 0. Overlap is resolved by winner-take-all.

In [ ]:
import os
import sys
import subprocess
import logging
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

# ── repo paths ────────────────────────────────────────────────────────────────
REPO_ROOT     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(REPO_ROOT))
from paths import ROOT as MOVIE_ROOT
CF_DIR        = REPO_ROOT / "cf_modeling"
VICSOMPY_REPO = MOVIE_ROOT / "Vicarious_somatotopy"

for p in [str(REPO_ROOT), str(CF_DIR), str(VICSOMPY_REPO)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── pycortex — must propagate new Database to all submodules ──────────────────
PYCORTEX_STORE = str(MOVIE_ROOT / "data" / "hedger2026")
os.environ["PYCORTEX_FILESTORE"] = PYCORTEX_STORE

import cortex
import cortex.database
new_db = cortex.database.Database(PYCORTEX_STORE)
cortex.db = new_db
cortex.database.default_filestore = new_db
for _mod_name, _mod in sys.modules.items():
    if _mod_name.startswith("cortex") and hasattr(_mod, "db"):
        _mod.db = new_db
cortex.options.config.set("basic", "filestore", PYCORTEX_STORE)
CX_SUB = list(cortex.db.subjects.keys())[0]

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s  %(levelname)s  %(message)s",
                    datefmt="%H:%M:%S")

print(f"Pycortex subject: {CX_SUB}")
print("Imports OK")

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================

DATA_BASE      = MOVIE_ROOT / "data"
OUTPUT_BASE    = MOVIE_ROOT / "outputs" / "cf_modeling"
MASKS_DIR      = OUTPUT_BASE / "masks"
MASKS_DIR.mkdir(parents=True, exist_ok=True)

GLASSER_DLABEL = str(
    DATA_BASE / "HCP_S1200_GroupAvg_v1" /
    "Q1-Q6_RelatedParcellation210"
    ".CorticalAreas_dil_Final_Final_Areas_Group_Colors"
    ".59k_fs_LR.dlabel.nii"
)

PREPROCESSED_DIR = DATA_BASE / "preprocessed" / "average_sub" / "sg_psc"
FMRI_SUFFIX      = "sg_psc"
TEMPLATE_CIFTI   = str(
    PREPROCESSED_DIR /
    f"group_average_{FMRI_SUFFIX}_cortex_59k.dtseries.nii"
)

# ── Source run: A1 × V1 ───────────────────────────────────────────────────────
SRC_ROI_A = "A1"
SRC_ROI_B = "V1"
MODE      = "group_average"

src_roi_tag  = f"{SRC_ROI_A}_{SRC_ROI_B}"
src_prep_dir = OUTPUT_BASE / MODE / src_roi_tag / "prep"

# ── Functional mask names ─────────────────────────────────────────────────────
OUT_ROI_A = "auditory_cx"    # derived from SRC_ROI_A (A1)
OUT_ROI_B = "visual_cx"      # derived from SRC_ROI_B (V1)

# Threshold settings
THRESHOLD_METHOD = "percentile"  # "percentile" | "absolute"
THRESHOLD_Q      = 99.0          #  percentile of R2_nc among valid verts
OVERLAP_METHOD   = "winner"      # "winner" | "allow" | "exclude"
MIN_R2_FULL      = 0.0           # only keep vertices with R2_full > 0

# ── Second CF run: auditory_cx × visual_cx ───────────────────────────────────
FUN_ROI_A    = OUT_ROI_A
FUN_ROI_B    = OUT_ROI_B
fun_roi_tag  = f"{FUN_ROI_A}_{FUN_ROI_B}"
fun_prep_dir = OUTPUT_BASE / MODE / fun_roi_tag / "prep"

N_LBOE   = 200
BACKEND  = "torch_cuda"
N_ITER   = 20
N_TARGETS_BATCH = 20000

print("Config:")
print(f"  Source pair     : {SRC_ROI_A} × {SRC_ROI_B}")
print(f"  Functional pair : {FUN_ROI_A} × {FUN_ROI_B}")
print(f"  Threshold       : {THRESHOLD_METHOD} q={THRESHOLD_Q}")
print(f"  Overlap         : {OVERLAP_METHOD}")
print(f"  Template CIFTI  : {TEMPLATE_CIFTI}")

## Step 1: Source CF Run (A1 × V1)

Load results from disk if already computed; otherwise run the full pipeline.

In [ ]:
import subprocess

CONDA_ENV = "movie"

def run_script(script_path, *args):
    """Run a Python script in the movie conda environment."""
    cmd = ["conda", "run", "--no-capture-output", "-n", CONDA_ENV,
           "python", str(script_path)] + list(args)
    print(f"  Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=False, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Script failed: {script_path}")


# ── Check if source run is already done ───────────────────────────────────────
src_done = (
    (src_prep_dir / f"R2_{SRC_ROI_A}_nc.npy").exists() and
    (src_prep_dir / f"R2_{SRC_ROI_B}_nc.npy").exists()
)

if src_done:
    print(f"Source run {SRC_ROI_A}×{SRC_ROI_B} already done — loading results.")
else:
    print(f"Source run {SRC_ROI_A}×{SRC_ROI_B} not found — running pipeline.")

    # Step 01: geometry
    print("\n[01] Building subsurfaces + LBOEs ...")
    run_script(
        CF_DIR / "01_extract_geometry.py",
        "--mode",            MODE,
        "--roi-a",           SRC_ROI_A,
        "--roi-b",           SRC_ROI_B,
        "--n-lboe",          str(N_LBOE),
        "--pycortex-store",  PYCORTEX_STORE,
        "--glasser-dlabel",  GLASSER_DLABEL,
        "--masks-dir",       str(MASKS_DIR),
        "--output-base",     str(OUTPUT_BASE),
        "--vicsompy-repo",   str(VICSOMPY_REPO),
    )

    # Step 02: fit CF model
    print("\n[02] Fitting CF model ...")
    run_script(
        CF_DIR / "02_fit_cf_model.py",
        "--mode",              MODE,
        "--roi-a",             SRC_ROI_A,
        "--roi-b",             SRC_ROI_B,
        "--preprocessed-dir", str(PREPROCESSED_DIR),
        "--fmri-suffix",       FMRI_SUFFIX,
        "--template-cifti",    TEMPLATE_CIFTI,
        "--output-base",       str(OUTPUT_BASE),
        "--backend",           BACKEND,
        "--n-iter",            str(N_ITER),
        "--n-targets-batch",   str(N_TARGETS_BATCH),
        "--vicsompy-repo",     str(VICSOMPY_REPO),
    )

    print("\nSource pipeline complete.")

print(f"prep dir: {src_prep_dir}")

In [ ]:
# ── Load source R² maps ───────────────────────────────────────────────────────
R2_full_src = np.load(src_prep_dir / "R2_full.npy")
R2_a_nc_src = np.load(src_prep_dir / f"R2_{SRC_ROI_A}_nc.npy")
R2_b_nc_src = np.load(src_prep_dir / f"R2_{SRC_ROI_B}_nc.npy")
R2_a_src    = np.load(src_prep_dir / f"R2_{SRC_ROI_A}.npy")
R2_b_src    = np.load(src_prep_dir / f"R2_{SRC_ROI_B}.npy")

print(f"Maps loaded. Shape: {R2_full_src.shape}")
print(f"  R2_full:        mean={R2_full_src.mean():.4f}  frac>0={np.mean(R2_full_src>0):.1%}")
print(f"  R2_{SRC_ROI_A}_nc:  mean={R2_a_nc_src.mean():.4f}  frac>0={np.mean(R2_a_nc_src>0):.1%}")
print(f"  R2_{SRC_ROI_B}_nc:  mean={R2_b_nc_src.mean():.4f}  frac>0={np.mean(R2_b_nc_src>0):.1%}")

## Step 2: Define Functional Masks

Threshold the null-corrected R² maps to create auditory and visual cortex masks.

In [ ]:
# ── Valid vertices: R2_full > 0 ───────────────────────────────────────────────
valid = R2_full_src > MIN_R2_FULL
n_valid = int(valid.sum())
print(f"Valid vertices (R2_full > {MIN_R2_FULL}): {n_valid} / {len(valid)}  "
      f"({100*valid.mean():.1f}%)")

# ── Threshold ─────────────────────────────────────────────────────────────────
if THRESHOLD_METHOD == "percentile":
    thr_a = max(0.0, float(np.percentile(R2_a_nc_src[valid], THRESHOLD_Q)))
    thr_b = max(0.0, float(np.percentile(R2_b_nc_src[valid], THRESHOLD_Q)))
else:
    thr_a = thr_b = THRESHOLD_Q

print(f"\nThresholds ({THRESHOLD_METHOD} q={THRESHOLD_Q}):")
print(f"  {SRC_ROI_A}: {thr_a:.4f}")
print(f"  {SRC_ROI_B}: {thr_b:.4f}")

raw_mask_a = valid & (R2_a_nc_src > thr_a)
raw_mask_b = valid & (R2_b_nc_src > thr_b)
print(f"\nRaw mask sizes:")
print(f"  {OUT_ROI_A}: {raw_mask_a.sum()} verts  ({100*raw_mask_a.sum()/n_valid:.1f}% of valid)")
print(f"  {OUT_ROI_B}: {raw_mask_b.sum()} verts  ({100*raw_mask_b.sum()/n_valid:.1f}% of valid)")

overlap = raw_mask_a & raw_mask_b
print(f"  Overlap: {overlap.sum()} verts")

In [ ]:
# ── Overlap resolution ────────────────────────────────────────────────────────
mask_a = raw_mask_a.copy()
mask_b = raw_mask_b.copy()

if OVERLAP_METHOD == "winner" and overlap.sum() > 0:
    a_wins = overlap & (R2_a_nc_src >= R2_b_nc_src)
    b_wins = overlap & (R2_b_nc_src > R2_a_nc_src)
    mask_a = (mask_a & ~overlap) | a_wins
    mask_b = (mask_b & ~overlap) | b_wins
    print(f"winner: {a_wins.sum()} → {OUT_ROI_A},  {b_wins.sum()} → {OUT_ROI_B}")
elif OVERLAP_METHOD == "exclude" and overlap.sum() > 0:
    mask_a = mask_a & ~overlap
    mask_b = mask_b & ~overlap
    print(f"exclude: removed {overlap.sum()} from both")
else:
    print(f"allow: {overlap.sum()} vertices kept in both masks")

print(f"\nFinal mask sizes:")
print(f"  {OUT_ROI_A}: {mask_a.sum()} grayord verts")
print(f"  {OUT_ROI_B}: {mask_b.sum()} grayord verts")
print(f"  Remaining overlap: {(mask_a & mask_b).sum()}")

In [ ]:
# ── Project to sphere hemispheres and save CSV masks ─────────────────────────
from lib.data_adapter import grayord_to_sphere_space, N_VERTS_PER_HEM
import pandas as pd

bm_axis = nib.load(TEMPLATE_CIFTI).header.get_axis(1)

def grayord_to_csv_masks(mask_grayord, roi_name, masks_dir):
    """Project grayordinate bool mask → sphere → L/R CSV files."""
    sphere = grayord_to_sphere_space(mask_grayord.astype(np.float32), bm_axis)
    mask_L = sphere[:N_VERTS_PER_HEM] > 0.5
    mask_R = sphere[N_VERTS_PER_HEM:] > 0.5
    for hem, mask in [("L", mask_L), ("R", mask_R)]:
        out = os.path.join(masks_dir, f"{roi_name}_{hem}_mask.csv")
        pd.DataFrame({"mask": mask.astype(bool)}).to_csv(out, index=False)
        print(f"  Saved: {os.path.basename(out)}  ({mask.sum()} True / {len(mask)} total)")
    return mask_L, mask_R

masks_dir_str = str(MASKS_DIR)

print(f"Saving CSV masks to: {masks_dir_str}")
mask_a_L, mask_a_R = grayord_to_csv_masks(mask_a, OUT_ROI_A, masks_dir_str)
mask_b_L, mask_b_R = grayord_to_csv_masks(mask_b, OUT_ROI_B, masks_dir_str)

print(f"\nSphere-space counts:")
print(f"  {OUT_ROI_A}: L={mask_a_L.sum()}  R={mask_a_R.sum()}")
print(f"  {OUT_ROI_B}: L={mask_b_L.sum()}  R={mask_b_R.sum()}")

In [ ]:
# ── Quick flatmap preview of the masks (pycortex) ─────────────────────────────
# grayord_to_sphere_space and N_VERTS_PER_HEM already imported in prev cell

def to_sphere(grayord_arr):
    """Map (59412,) grayord array → (118584,) sphere, zeros at medial wall."""
    return grayord_to_sphere_space(grayord_arr.astype(np.float32), bm_axis)

# Mask preview in sphere space for pycortex
sph_mask_a = to_sphere(mask_a.astype(np.float32))
sph_mask_b = to_sphere(mask_b.astype(np.float32))

# Combined map: 1=auditory, 2=visual, 0=neither
sph_combined = sph_mask_a + 2 * sph_mask_b

vol_a    = cortex.Volume(sph_mask_a,   CX_SUB, "fiducial", vmin=0, vmax=1, cmap="Reds")
vol_b    = cortex.Volume(sph_mask_b,   CX_SUB, "fiducial", vmin=0, vmax=1, cmap="Blues")
vol_comb = cortex.Volume(sph_combined, CX_SUB, "fiducial", vmin=0, vmax=3, cmap="tab10")

fig, axes = plt.subplots(1, 3, figsize=(21, 5))
cortex.quickflat.make_figure(vol_a,    with_rois=False, fig=fig, ax=axes[0])
cortex.quickflat.make_figure(vol_b,    with_rois=False, fig=fig, ax=axes[1])
cortex.quickflat.make_figure(vol_comb, with_rois=False, fig=fig, ax=axes[2])

axes[0].set_title(f"{OUT_ROI_A} mask (from {SRC_ROI_A} R²_nc > {thr_a:.3f})", fontsize=11)
axes[1].set_title(f"{OUT_ROI_B} mask (from {SRC_ROI_B} R²_nc > {thr_b:.3f})", fontsize=11)
axes[2].set_title(f"Combined: red={OUT_ROI_A}, blue={OUT_ROI_B}", fontsize=11)
plt.suptitle(f"Functional masks — {THRESHOLD_METHOD} q={THRESHOLD_Q}  overlap={OVERLAP_METHOD}",
             fontsize=13)
plt.tight_layout()
plt.savefig(str(MASKS_DIR / f"functional_masks_{OUT_ROI_A}_{OUT_ROI_B}.png"),
            dpi=150, bbox_inches="tight")
plt.show()

## Step 3: Second CF Run (auditory_cx × visual_cx)

In [ ]:
# ── Check if functional run is already done ───────────────────────────────────
fun_done = (
    (fun_prep_dir / f"R2_{FUN_ROI_A}_nc.npy").exists() and
    (fun_prep_dir / f"R2_{FUN_ROI_B}_nc.npy").exists()
)

if fun_done:
    print(f"Functional run {FUN_ROI_A}×{FUN_ROI_B} already done — loading results.")
else:
    print(f"Functional run {FUN_ROI_A}×{FUN_ROI_B} not found — running pipeline.")

    # Step 01: geometry for functional pair (uses CSV masks)
    print("\n[01] Building subsurfaces + LBOEs for functional masks ...")
    run_script(
        CF_DIR / "01_extract_geometry.py",
        "--mode",            MODE,
        "--roi-a",           FUN_ROI_A,
        "--roi-b",           FUN_ROI_B,
        "--n-lboe",          str(N_LBOE),
        "--pycortex-store",  PYCORTEX_STORE,
        "--glasser-dlabel",  GLASSER_DLABEL,
        "--masks-dir",       str(MASKS_DIR),   # <-- CSV fallback for functional ROIs
        "--output-base",     str(OUTPUT_BASE),
        "--vicsompy-repo",   str(VICSOMPY_REPO),
    )

    # Step 02: fit CF model
    print("\n[02] Fitting CF model (functional masks) ...")
    run_script(
        CF_DIR / "02_fit_cf_model.py",
        "--mode",              MODE,
        "--roi-a",             FUN_ROI_A,
        "--roi-b",             FUN_ROI_B,
        "--preprocessed-dir", str(PREPROCESSED_DIR),
        "--fmri-suffix",       FMRI_SUFFIX,
        "--template-cifti",    TEMPLATE_CIFTI,
        "--output-base",       str(OUTPUT_BASE),
        "--backend",           BACKEND,
        "--n-iter",            str(N_ITER),
        "--n-targets-batch",   str(N_TARGETS_BATCH),
        "--vicsompy-repo",     str(VICSOMPY_REPO),
    )

    print("\nFunctional CF pipeline complete.")

In [ ]:
# ── Integration maps for functional run ───────────────────────────────────────
fun_cifti_dir = OUTPUT_BASE / MODE / fun_roi_tag / "cifti_maps"
if not fun_cifti_dir.exists() or not list(fun_cifti_dir.glob("*.nii")):
    print("Running integration_maps.py ...")
    run_script(
        CF_DIR / "integration_maps.py",
        "--mode",           MODE,
        "--roi-a",          FUN_ROI_A,
        "--roi-b",          FUN_ROI_B,
        "--output-base",    str(OUTPUT_BASE),
        "--template-cifti", TEMPLATE_CIFTI,
    )
else:
    print("Integration maps already exist — skipping.")

# ── Load functional run R² maps ───────────────────────────────────────────────
R2_full_fun = np.load(fun_prep_dir / "R2_full.npy")
R2_a_nc_fun = np.load(fun_prep_dir / f"R2_{FUN_ROI_A}_nc.npy")
R2_b_nc_fun = np.load(fun_prep_dir / f"R2_{FUN_ROI_B}_nc.npy")

_prod_nc_fun_path = fun_prep_dir / "product_map_nc.npy"
if _prod_nc_fun_path.exists():
    prod_nc_fun = np.load(_prod_nc_fun_path)
else:
    prod_nc_fun = np.sqrt(np.clip(R2_a_nc_fun, 0, None) * np.clip(R2_b_nc_fun, 0, None)).astype(np.float32)
    print("  product_map_nc (computed inline)")

print(f"\nFunctional run maps ({FUN_ROI_A} × {FUN_ROI_B}):")
print(f"  R2_full:         mean={R2_full_fun.mean():.4f}  frac>0={np.mean(R2_full_fun>0):.1%}")
print(f"  R2_{FUN_ROI_A}_nc: mean={R2_a_nc_fun.mean():.4f}  frac>0={np.mean(R2_a_nc_fun>0):.1%}")
print(f"  R2_{FUN_ROI_B}_nc: mean={R2_b_nc_fun.mean():.4f}  frac>0={np.mean(R2_b_nc_fun>0):.1%}")
print(f"  product_map_nc:  mean={prod_nc_fun.mean():.4f}  frac>0={np.mean(prod_nc_fun>0):.1%}")

## Step 4: Comparison — A1×V1 vs auditory_cx×visual_cx

In [ ]:
# ── Convert all maps to sphere space for pycortex ─────────────────────────────
sph_a_nc_src  = to_sphere(R2_a_nc_src)
sph_b_nc_src  = to_sphere(R2_b_nc_src)
sph_full_src  = to_sphere(R2_full_src)

sph_a_nc_fun  = to_sphere(R2_a_nc_fun)
sph_b_nc_fun  = to_sphere(R2_b_nc_fun)
sph_full_fun  = to_sphere(R2_full_fun)
sph_prod_fun  = to_sphere(prod_nc_fun)

# ── Shared vmax ───────────────────────────────────────────────────────────────
pos_vals = np.concatenate([
    sph_a_nc_src[sph_a_nc_src > 0],
    sph_b_nc_src[sph_b_nc_src > 0],
    sph_a_nc_fun[sph_a_nc_fun > 0],
    sph_b_nc_fun[sph_b_nc_fun > 0],
])
vmax = min(float(np.percentile(pos_vals, 95)), 0.15) if len(pos_vals) > 0 else 0.1
print(f"Shared vmax: {vmax:.4f}")

In [ ]:
# ── 2D colormaps: Fig 3a comparison ──────────────────────────────────────────
from vicsompy.vis import Plot
from vicsompy.utils import load_plot_pkg_yaml

def make_2d_plot(sph_b, sph_a, title, vmax):
    """Hedger Fig 3a style 2D colormap. dat=visual(blue), dat2=audio(red)."""
    sph_b_pos = np.clip(sph_b, 0, None)
    sph_a_pos = np.clip(sph_a, 0, None)
    plotviews = load_plot_pkg_yaml()['plotviews']
    cfg_2d = plotviews['variance_decomp_full'].copy()
    cfg_2d.update({
        'vmax': vmax, 'vmax2': vmax,
        'vmin': 0.0,  'vmin2': 0.0,
        'with_colorbar': True,
        'with_rois': False,
        'subject': CX_SUB,
    })
    cfg_2d.pop('ROIS', None)
    plotter = Plot(**cfg_2d)
    fig = plotter.uber_plot(dat=sph_b_pos, dat2=sph_a_pos)
    fig.suptitle(title, fontsize=13)
    return fig


# Source run: A1 × V1
fig_src = make_2d_plot(
    sph_b_nc_src, sph_a_nc_src,
    title=f"Source run — {SRC_ROI_A}×{SRC_ROI_B}: blue={SRC_ROI_B} R²_nc, red={SRC_ROI_A} R²_nc",
    vmax=vmax,
)
fig_src.savefig(str(OUTPUT_BASE / MODE / src_roi_tag / "cifti_maps" / "fig3a_src.png"),
                dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Functional run: auditory_cx × visual_cx
fig_fun = make_2d_plot(
    sph_b_nc_fun, sph_a_nc_fun,
    title=f"Functional run — {FUN_ROI_A}×{FUN_ROI_B}: blue={FUN_ROI_B} R²_nc, red={FUN_ROI_A} R²_nc",
    vmax=vmax,
)
fig_fun.savefig(str(OUTPUT_BASE / MODE / fun_roi_tag / "cifti_maps" / "fig3a_fun.png"),
                dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Side-by-side 1D R²_nc maps ────────────────────────────────────────────────
maps_to_compare = [
    (sph_a_nc_src,  f"{SRC_ROI_A} R²_nc (source)",   "Reds_r"),
    (sph_a_nc_fun,  f"{FUN_ROI_A} R²_nc (functional)", "Reds_r"),
    (sph_b_nc_src,  f"{SRC_ROI_B} R²_nc (source)",   "Blues_r"),
    (sph_b_nc_fun,  f"{FUN_ROI_B} R²_nc (functional)", "Blues_r"),
    (sph_full_src,  "R²_full (source)",                "viridis"),
    (sph_full_fun,  "R²_full (functional)",             "viridis"),
]

fig, axes = plt.subplots(3, 2, figsize=(22, 18))
for ax, (arr, title, cmap) in zip(axes.flat, maps_to_compare):
    vol = cortex.Volume(arr, CX_SUB, "fiducial",
                        vmin=0, vmax=vmax, cmap=cmap)
    cortex.quickflat.make_figure(vol, with_rois=False, fig=fig, ax=ax)
    ax.set_title(title, fontsize=10)

plt.suptitle("Source (A1×V1) vs Functional (auditory_cx×visual_cx) — R²_nc maps",
             fontsize=13)
plt.tight_layout()
plt.savefig(str(OUTPUT_BASE / MODE / fun_roi_tag / "cifti_maps" / "comparison_maps.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Integration score (product_nc) comparison ────────────────────────────────
_prod_nc_src_path = src_prep_dir / "product_map_nc.npy"
if _prod_nc_src_path.exists():
    prod_nc_src = np.load(_prod_nc_src_path)
else:
    prod_nc_src = np.sqrt(np.clip(R2_a_nc_src, 0, None) * np.clip(R2_b_nc_src, 0, None)).astype(np.float32)
    print("  product_map_nc src (computed inline)")

sph_prod_src = to_sphere(prod_nc_src)

vmax_prod = min(float(np.percentile(
    np.concatenate([sph_prod_src[sph_prod_src > 0], sph_prod_fun[sph_prod_fun > 0]]),
    95
)), 0.1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (arr, title) in zip(axes, [
    (sph_prod_src, f"Integration score NC\n{SRC_ROI_A}×{SRC_ROI_B} (source)"),
    (sph_prod_fun, f"Integration score NC\n{FUN_ROI_A}×{FUN_ROI_B} (functional)"),
]):
    vol = cortex.Volume(arr, CX_SUB, "fiducial", vmin=0, vmax=vmax_prod, cmap="hot_r")
    cortex.quickflat.make_figure(vol, with_rois=False, fig=fig, ax=ax)
    ax.set_title(title, fontsize=11)

plt.suptitle("NC Integration score: √(R²_a_nc × R²_b_nc)", fontsize=13)
plt.tight_layout()
plt.savefig(str(OUTPUT_BASE / MODE / fun_roi_tag / "cifti_maps" / "integration_comparison.png"),
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
import pandas as pd

rows = []
for run_name, r2f, r2a, r2b, prod in [
    (f"{SRC_ROI_A}×{SRC_ROI_B}",  R2_full_src, R2_a_nc_src, R2_b_nc_src, prod_nc_src),
    (f"{FUN_ROI_A}×{FUN_ROI_B}", R2_full_fun, R2_a_nc_fun, R2_b_nc_fun, prod_nc_fun),
]:
    rows.append({
        "Run":             run_name,
        "R2_full mean":    f"{r2f.mean():.4f}",
        "R2_full >0 (%)": f"{100*np.mean(r2f>0):.1f}",
        "R2_A_nc mean":    f"{r2a.mean():.4f}",
        "R2_A_nc >0 (%)": f"{100*np.mean(r2a>0):.1f}",
        "R2_B_nc mean":    f"{r2b.mean():.4f}",
        "R2_B_nc >0 (%)": f"{100*np.mean(r2b>0):.1f}",
        "prod_nc mean":    f"{prod.mean():.4f}",
        "prod_nc >0 (%)": f"{100*np.mean(prod>0):.1f}",
    })

df = pd.DataFrame(rows).set_index("Run")
print("\nSummary comparison:")
display(df.T)